In [1]:
import glob
import os

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

C:\Users\AsusIran\AppData\Local\Temp\ipykernel_33824\2819249026.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
d:\agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

True

In [3]:
api = os.getenv("GAPGPT_API_KEY")
url = os.getenv("GAPGPT_BASE_URL")

embedding_model = "all-MiniLM-L6-v2"
DB_name = "ch_database"
embedding = HuggingFaceEmbeddings(model_name=embedding_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4474.69it/s]


In [4]:
model_name = "gpt-5-nano"

In [5]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool


llm = ChatOpenAI(
    model=model_name,
    base_url=url,
    api_key=api
)



In [16]:
# اسناد را همیشه بارگذاری و chunk می‌کنیم؛ BM25 به chunkها نیاز دارد.
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    for doc in loader.load():
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

# Chroma را فقط در صورت نبودن دیتابیس می‌سازیم.
if os.path.exists(DB_name) and os.listdir(DB_name):
    print("📁 Loading existing Chroma vector store...")
    vectorstore = Chroma(
        persist_directory=DB_name, embedding_function=embedding
    )
else:
    print("⚡ Creating new Chroma vector store from markdown files...")
    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embedding, persist_directory=DB_name
    )

📁 Loading existing Chroma vector store...


In [7]:
# Company, product, contract, and employee documents are available to the assistant.
ALLOWED_DOC_TYPES = ["company", "products", "contracts" , "employees"]
SENSITIVE_QUERY_TERMS = {
    "salary", "compensation", "bonus", "performance review",
    "date of birth", "birthday", "home address", "phone number",
}
INSURELLM_TERMS = {
    "insurellm", "carllm", "homellm", "lifellm", "healthllm",
    "bizllm", "markellm", "claimllm", "rellm",  "products", "company", "products", "contracts" , "employees", 
}
OFF_TOPIC_RESPONSE = "I can only answer questions related to Insurellm."
SENSITIVE_RESPONSE = (
    "I can help with Insurellm's public company, product, and contract information, "
    "but I cannot provide employee personal or compensation information."
)

In [8]:
# Query expansion entirely uses the LangChain chat model configured above.
def generate_queries(query: str) -> list[str]:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a search query optimizer for the Insurellm knowledge base.
Generate exactly 3 alternative search queries. Preserve the exact intent; do not add or reinterpret entities.
Return only the queries, one per line, and do not answer the question."""),
        ("human", "{query}"),
    ])
    response = (prompt | llm | StrOutputParser()).invoke({"query": query})

    return [
        line.strip()
        for line in response.splitlines()
        if line.strip()
    ]

In [9]:
from pydantic import BaseModel, Field


# ---------------------------
# Graph extraction schemas
# ---------------------------

class Entity(BaseModel):
    name: str
    type: str


class Relationship(BaseModel):
    source: str
    type: str
    target: str


class GraphExtraction(BaseModel):
    entities: list[Entity] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)


# ---------------------------
# Allowed graph schema
# ---------------------------

ALLOWED_ENTITY_TYPES = {
    "Employee",
    "Company",
    "Product",
    "Contract",
    "JobPosition",
    "Department",
    "Location",
    "Project",
    "Document",
}

ALLOWED_RELATIONSHIP_TYPES = {
    "WORKS_FOR",
    "HAS_ROLE",
    "LOCATED_IN",
    "WORKED_ON",
    "MENTIONED_IN",
    "OFFERS",
    "HAS_CONTRACT",
    "POSTS",
    "IN_DEPARTMENT",
    "WITH",
    "FOR_PRODUCT",
}


# ---------------------------
# Graph extractor agent
# ---------------------------

graph_parser = PydanticOutputParser(pydantic_object=GraphExtraction)
graph_prompt = ChatPromptTemplate.from_messages([
    ("system", f"""
You extract structured knowledge graph information from company documents.

Your task:
1. Identify meaningful entities.
2. Identify meaningful relationships between those entities.
3. Return ONLY information explicitly supported by the document.
4. Do NOT invent entities or relationships.
5. Use only the allowed entity types.
6. Use only the allowed relationship types.
7. Keep entity names consistent.
8. Do not create nodes for simple scalar facts such as salary, date of birth,
   contract amount, performance score, or dates unless the schema explicitly
   requires them as entities.

Allowed entity types:
{sorted(ALLOWED_ENTITY_TYPES)}

Allowed relationship types:
{sorted(ALLOWED_RELATIONSHIP_TYPES)}

Important:
- Employees are valid entities and must NOT be ignored.
- A company, product, contract, employee, job position, department,
  location, or project should be extracted when clearly supported.
- Do not infer relationships merely because two entities appear in the
  same document.
    
{{format_instructions}}
"""),
    ("human", "Extract the knowledge graph from this document:\n\n{document}"),
])
graph_extractor = graph_prompt | llm | graph_parser

In [10]:
def extract_graph(text: str) -> GraphExtraction:
    return graph_extractor.invoke({
        "document": text,
        "format_instructions": graph_parser.get_format_instructions(),
    })

In [66]:
text = """
# HR Record

# Oliver Spencer

## Summary
- **Date of Birth**: May 14, 1990
- **Job Title**: Backend Software Engineer
- **Location**: Austin, Texas
- **Current Salary**: $125,000

## Insurellm Career Progression
- **March 2018**: Joined Insurellm as a Backend Developer I.
- **July 2019**: Promoted to Backend Developer II.
- **June 2021**: Transitioned to Backend Software Engineer.
- **September 2022**: Assigned as the lead engineer for the new "Innovate" initiative.
"""


graph = extract_graph(text)

print(graph.model_dump_json(indent=2))

{
  "entities": [
    {
      "name": "Oliver Spencer",
      "type": "Employee"
    },
    {
      "name": "Insurellm",
      "type": "Company"
    },
    {
      "name": "Backend Software Engineer",
      "type": "JobPosition"
    },
    {
      "name": "Backend Developer I",
      "type": "JobPosition"
    },
    {
      "name": "Backend Developer II",
      "type": "JobPosition"
    },
    {
      "name": "Austin, Texas",
      "type": "Location"
    },
    {
      "name": "Innovate",
      "type": "Project"
    },
    {
      "name": "HR Record",
      "type": "Document"
    }
  ],
  "relationships": [
    {
      "source": "Oliver Spencer",
      "type": "WORKS_FOR",
      "target": "Insurellm"
    },
    {
      "source": "Oliver Spencer",
      "type": "HAS_ROLE",
      "target": "Backend Software Engineer"
    },
    {
      "source": "Oliver Spencer",
      "type": "HAS_ROLE",
      "target": "Backend Developer I"
    },
    {
      "source": "Oliver Spencer",
      "type": "

In [18]:
# تمام دسته‌های مجاز، شامل اسناد کارکنان، وارد هر دو موتور جست‌وجو می‌شوند.
public_chunks = [
    doc for doc in chunks
    if doc.metadata.get("doc_type") in ALLOWED_DOC_TYPES
]

# مسیر اول Hybrid Search: بازیابی معنایی با Chroma.
public_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {"doc_type": {"$in": ALLOWED_DOC_TYPES}},
    },
)

# مسیر دوم Hybrid Search: بازیابی واژه‌محور با BM25.
bm25 = BM25Okapi([
    doc.page_content.casefold().split()
    for doc in public_chunks
])

print(f"Total chunks: {len(chunks)}")
print(f"Public chunks: {len(public_chunks)}")

Total chunks: 413
Public chunks: 413


In [17]:
def bm25_search(query: str, top_k: int = 5):
    
    scores = bm25.get_scores(query.casefold().split())
    top_indices = scores.argsort()[::-1][:top_k]

    return [
        public_chunks[idx]
        for idx in top_indices
        if scores[idx] > 0
    ]

In [19]:
def reciprocal_rank_fusion(ranked_results, k=60):
   
    scores = {}
    unique_documents = {}

    for results in ranked_results:
        for rank, doc in enumerate(results, start=1):
            # source به‌تنهایی کافی نیست؛ هر فایل چند chunk دارد.
            key = (doc.metadata.get("source", ""), doc.page_content)
            scores[key] = scores.get(key, 0.0) + 1 / (k + rank)
            unique_documents[key] = doc

    ranked_keys = sorted(scores, key=scores.get, reverse=True)
    return [unique_documents[key] for key in ranked_keys]

In [20]:
# بعد از ادغام RRF، CrossEncoder بهترین chunkها را دوباره رتبه‌بندی می‌کند.
reranker = CrossEncoder("BAAI/bge-reranker-base")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5410.20it/s]


In [21]:
def rerank_documents(query, docs, top_k=3):
    """انتخاب مرتبط‌ترین نتایج Hybrid Search برای پاسخ نهایی."""
    if not docs:
        return []

    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(zip(docs, scores), key=lambda item: item[1], reverse=True)
    return [doc for doc, _ in scored_docs[:top_k]]

In [22]:
def hybrid_search(query: str, alternative_queries: list[str] | None = None):
    """اجرای Vector Search و BM25 و ادغام خروجی آن‌ها با RRF."""
    ranked_results = []
    for search_query in [query, *(alternative_queries or [])]:
        ranked_results.append(public_retriever.invoke(search_query))
        ranked_results.append(bm25_search(search_query))

    return reciprocal_rank_fusion(ranked_results)

In [24]:
from langchain_neo4j import Neo4jGraph

NEO4J_URI =  "bolt://localhost:8687"
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD" )


graph_db = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD
)

print(graph_db.query("RETURN 'Connected!' AS status"))

[{'status': 'Connected!'}]


In [86]:
graph_db.query("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE;")

[]

In [87]:
def populate_graph(documents: list):
    cypher_query = """
    UNWIND $entities AS entity
    MERGE (e:Entity {name: entity.name})
    SET e.type = entity.type

    WITH 1 as dummy
    UNWIND $relationships AS rel
    MERGE (source:Entity {name: rel.source})
    MERGE (target:Entity {name: rel.target})
    MERGE (source)-[r:RELATION {type: rel.type}]->(target)
    """

    print(f"🔄 Processing {len(documents)} documents for Graph Ingestion...")
    for idx, doc in enumerate(documents, 1):
        try:
            extraction: GraphExtraction = extract_graph(doc.page_content)
            
            entities_payload = [e.model_dump() for e in extraction.entities]
            rels_payload = [r.model_dump() for r in extraction.relationships]
            
            if entities_payload or rels_payload:
                graph_db.query(
                    cypher_query,
                    params={
                        "entities": entities_payload,
                        "relationships": rels_payload
                    }
                )
            if idx % 10 == 0 or idx == len(documents):
                print(f"✅ Processed {idx}/{len(documents)} docs")
        except Exception as e:
            print(f"⚠️ Error processing doc {idx}: {e}")

# اجرای بارگذاری روی چانک‌های مجاز
populate_graph(public_chunks)

🔄 Processing 413 documents for Graph Ingestion...
✅ Processed 10/413 docs
✅ Processed 20/413 docs
✅ Processed 30/413 docs
✅ Processed 40/413 docs
✅ Processed 50/413 docs
✅ Processed 60/413 docs
✅ Processed 70/413 docs
✅ Processed 80/413 docs
✅ Processed 90/413 docs
✅ Processed 100/413 docs
✅ Processed 110/413 docs
✅ Processed 120/413 docs
✅ Processed 130/413 docs
✅ Processed 140/413 docs
⚠️ Error processing doc 145: Error code: 403 - {'error': {'message': '预扣费额度失败, 用户剩余额度: ＄0.273154, 需要预扣费额度: ＄0.800000 (request id: 202608181158465535996218268d9d6QeZZsLkj)', 'type': 'new_api_error', 'param': '', 'code': 'insufficient_user_quota'}}
⚠️ Error processing doc 146: Error code: 403 - {'error': {'message': '预扣费额度失败, 用户剩余额度: ＄0.273154, 需要预扣费额度: ＄0.800000 (request id: 202608181158469666930358268d9d6G5oWa7ND)', 'type': 'new_api_error', 'param': '', 'code': 'insufficient_user_quota'}}
⚠️ Error processing doc 147: Error code: 403 - {'error': {'message': '预扣费额度失败, 用户剩余额度: ＄0.273154, 需要预扣费额度: ＄0.80000

In [25]:
# ۱. ساخت تابع جست‌وجو در گراف بر اساس داده‌های ذخیره‌شده تا سند ۱۴۰
def graph_subgraph_search(query: str) -> str:
    raw_entities = entity_extractor_chain.invoke({"query": query})
    entities = [e.strip() for e in raw_entities.split(",") if e.strip()]
    if not entities:
        return ""
    
    cypher = """
    MATCH (e:Entity)
    WHERE toLower(e.name) IN [x IN $entities | toLower(x)]
    MATCH (e)-[r:RELATION]-(target:Entity)
    RETURN e.name + ' [' + r.type + '] ' + target.name AS fact
    LIMIT 25
    """
    results = graph_db.query(cypher, params={"entities": entities})
    facts = [record["fact"] for record in results]
    return "\n".join(f"- {f}" for f in facts) if facts else ""

# ۲. تست وضعیت نودهای موجود در دیتابیس
count_res = graph_db.query("MATCH (n:Entity) RETURN count(n) AS total_entities")
print("تعداد نودهای ذخیره‌شده تا این لحظه:", count_res)

تعداد نودهای ذخیره‌شده تا این لحظه: [{'total_entities': 321}]


In [26]:
rel_count = graph_db.query("MATCH ()-[r:RELATION]->() RETURN count(r) AS total_relations")
print("تعداد کل روابط ذخیره‌شده:", rel_count)

تعداد کل روابط ذخیره‌شده: [{'total_relations': 324}]


In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# زنجیره استخراج انتیتی از کوئری کاربر
entity_extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract all key named entities (people, companies, products, roles, locations) from the query. Return them as comma-separated values only. If none found, return nothing."),
    ("human", "{query}")
])
entity_extractor_chain = entity_extract_prompt | llm | StrOutputParser()

def graph_subgraph_search(query: str) -> str:
    """پیدا کردن انتیتی‌ها و بازیابی روابط مرتبط از Neo4j"""
    try:
        raw_entities = entity_extractor_chain.invoke({"query": query})
        entities = [e.strip() for e in raw_entities.split(",") if e.strip()]
        
        if not entities:
            return ""
        
        cypher = """
        MATCH (e:Entity)
        WHERE toLower(e.name) IN [x IN $entities | toLower(x)]
        MATCH (e)-[r:RELATION]-(target:Entity)
        RETURN e.name + ' [' + r.type + '] ' + target.name AS fact
        LIMIT 25
        """
        
        results = graph_db.query(cypher, params={"entities": entities})
        facts = [record["fact"] for record in results]
        
        if facts:
            return "\n".join(f"- {f}" for f in facts)
    except Exception as e:
        print(f"⚠️ Graph search error: {e}")
    return ""



In [28]:
# سیاست پاسخ‌گویی قبل از ساخت ابزار اجرا می‌شود.
def question_policy(query: str) -> str | None:
    normalized = query.casefold()
    if any(term in normalized for term in SENSITIVE_QUERY_TERMS):
        return SENSITIVE_RESPONSE
    if not any(term in normalized for term in INSURELLM_TERMS):
        return OFF_TOPIC_RESPONSE
    return None

instructions="""
You are a specialized AI assistant strictly representing the company Insurellm.

RULES:
1. You MUST ALWAYS use the `search_knowledge_base` tool first for ANY question asked by the user.
2. You are ONLY allowed to answer questions related to Insurellm, its company information, products, contracts, and employees.
3. If the user asks general knowledge questions, off-topic questions (e.g., geography, general coding, weather, math), or anything NOT related to Insurellm, politely refuse to answer and state: "I can only answer questions related to Insurellm."
4. Do NOT use your pre-trained general knowledge to answer off-topic queries.

"""
@tool
def search_knowledge_base(query: str) -> str:
    """Search the Insurellm knowledge base (Text & Knowledge Graph) for relevant documents and relationships."""

    # ۱. گاردریل
    policy_response = question_policy(query)
    if policy_response:
        return policy_response

    # ۲. بازیابی ساختاریافته از گراف دانش
    graph_context = graph_subgraph_search(query)
    if graph_context:
        print("\n📊 Graph Facts Found:")
        print(graph_context)
    else:
        print("\n📊 Graph Facts: No direct entities/facts found.")

    # ۳. گسترش کوئری (Query Expansion)
    queries = generate_queries(query)
    print("\n🔍 Generated Alternative Queries:")
    for q in queries:
        print(f"  - {q}")

    # ۴. بازیابی متنی (Hybrid Search + RRF)
    docs = hybrid_search(query, queries)
    if not docs:
        print("\n⚠️ No documents found via Hybrid Search.")
        return graph_context if graph_context else "No relevant information found."

    docs = docs[:10]
    print("\n📁 Top Chunks after Hybrid Search (RRF):")
    for i, doc in enumerate(docs, start=1):
        print(f"  {i}. {doc.metadata.get('source')}")

    # ۵. رتبه‌بندی مجدد (Reranking)
    reranked_docs = rerank_documents(query, docs, top_k=3)
    print("\n🎯 Top 3 Chunks after Cross-Encoder Reranking:")
    for i, doc in enumerate(reranked_docs, start=1):
        print(f"  {i}. {doc.metadata.get('source')}")

    text_context = "\n\n".join(doc.page_content for doc in reranked_docs)

    # ۶. تلفیق کانتکست نهایی
    combined = []
    if graph_context:
        combined.append(f"### KNOWLEDGE GRAPH RELATIONSHIPS:\n{graph_context}")
    if text_context:
        combined.append(f"### DETAILED TEXT CONTEXT:\n{text_context}")

    return "\n\n".join(combined) if combined else "No relevant information found."


rag_agent = create_agent(
    model=llm,
    tools=[search_knowledge_base],
    system_prompt=instructions,
)

In [157]:
response = await rag_agent.ainvoke({
    "messages": [{"role": "user", "content": "Which companies have active contracts for the product Homellm, and what other products does Insurellm offer to them?"}]
})

print(response["messages"][-1].content)


📊 Graph Facts Found:
- Insurellm [WITH] Insurellm - GreenValley Insurance Contract
- Insurellm [HAS_CONTRACT] Agreement
- Insurellm [OFFERS] Customer Portal
- Insurellm [WITH] Contract with Greenstone Insurance for Homellm
- Insurellm [HAS_CONTRACT] Contract with GreenField Holdings for Markellm
- Insurellm [OFFERS] Climate Risk Analytics
- Insurellm [OFFERS] Enterprise Tier
- Insurellm [HAS_CONTRACT] Insurellm - Fortress Business Underwriters Contract
- Insurellm [WORKS_FOR] John Smith
- Insurellm [HAS_CONTRACT] Insurellm - EverGuard Insurance Rellm Contract
- Insurellm [WITH] Rellm Contract with EverGuard Insurance
- Insurellm [WITH] IG-2023-EG
- Insurellm [LOCATED_IN] 123 Innovation Drive, Tech City, USA
- Insurellm [OFFERS] Enterprise-level Support
- Insurellm [OFFERS] Regulatory Compliance Suite
- Insurellm [WITH] Insurellm - Continental Commercial Group Contract
- Insurellm [HAS_CONTRACT] Insurellm and BrightWay Solutions Contract
- Insurellm [HAS_CONTRACT] Insurellm - BrightWay

In [76]:
# Retrieval evaluation: edit this set as your knowledge base and expected answers evolve.
# The metric uses public documents only, matching the production retrieval policy.
evaluation_set = [
    {
        "query": "What products does Insurellm offer?",
        "expected_sources": {"overview.md", "about.md"},
    },
    {
        "query": "What are the features of Rellm?",
        "expected_sources": {"Rellm.md"},
    },
    {
        "query": "What does Claimllm do?",
        "expected_sources": {"Claimllm.md", "overview.md"},
    },
    {
        "query": "Which Insurellm contracts use Homellm?",
        "expected_sources": {"Homellm.md", "overview.md"},
    },
]

def evaluate_retrieval(examples, k=5):
    """Report Hit@k and MRR for a labelled set of retrieval queries."""
    hits = 0
    reciprocal_ranks = []

    for example in examples:
        # ارزیابی همان مسیر Hybrid را می‌سنجد، نه فقط Vector Search را.
        results = hybrid_search(example["query"])[:k]
        result_names = [os.path.basename(doc.metadata.get("source", "")) for doc in results]
        first_match = next(
            (rank for rank, name in enumerate(result_names, start=1)
             if name in example["expected_sources"]),
            None,
        )
        hits += first_match is not None
        reciprocal_ranks.append(1 / first_match if first_match else 0)
        print(f"{example['query']}\n  Sources: {result_names}\n  Match rank: {first_match}")

    total = len(examples)
    metrics = {
        f"Hit@{k}": hits / total if total else 0,
        "MRR": sum(reciprocal_ranks) / total if total else 0,
    }
    print(f"\nMetrics: {metrics}")
    return metrics

metrics = evaluate_retrieval(evaluation_set)


What products does Insurellm offer?
  Sources: ['overview.md', 'Contract with Greenstone Insurance for Homellm.md', 'about.md', 'about.md', 'Lifellm.md']
  Match rank: 1
What are the features of Rellm?
  Sources: ['Rellm.md', 'Contract with Apex Reinsurance for Rellm - AI-Powered Enterprise Reinsurance Solution.md', 'Contract with Belvedere Insurance for Markellm.md', 'overview.md', 'Rellm.md']
  Match rank: 1
What does Claimllm do?
  Sources: ['Claimllm.md', 'Claimllm.md', 'Contract with Rapid Claims Associates for Claimllm.md', 'Contract with Greenstone Insurance for Homellm.md', 'Contract with Premier Adjusters Inc. for Claimllm.md']
  Match rank: 1
Which Insurellm contracts use Homellm?
  Sources: ['overview.md', 'Homellm.md', 'overview.md', 'Contract with Belvedere Insurance for Markellm.md', 'Homellm.md']
  Match rank: 1

Metrics: {'Hit@5': 1.0, 'MRR': 1.0}
